In [2]:
from spark_session import get_spark
import pyspark.sql 
from pyspark.sql import functions as F
spark = get_spark('test')
spark.stop()
x = spark.sql("select * from gld")
x = x.withColumn('Yr', F.year(F.col("Date_")))
df_high = x.filter(F.col("Gld_Close")>2000)
df_high.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gld_highs")
df_high.show()


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\Users\test\AppData\Local\Programs\Python\Python310\lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "c:\Users\test\AppData\Local\Programs\Python\Python310\lib\socket.py", line 705, in readinto
    return self._sock.recv_into(b)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\test\AppData\Local\Programs\Python\Python310\lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "c:\Users\test\AppData\Local\Programs\Python\Python310\lib\site-packages\py4j\clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiv

Py4JError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext

In [2]:
import yfinance as yf
import numpy as np 
import pandas as pd 
import datetime
from spark_session import get_spark
from pyspark.sql import functions as F
from delta.tables import DeltaTable
TICKERS = {
    'GC=F':  'Gold',
    'SI=F':  'Silver',
    'PL=F':  'Platinum',
    'PA=F':  'Palladium',
    'HG=F':  'Copper',
    'CL=F':  'WTI Crude Oil',
    'BZ=F':  'Brent Crude',
    'XEL':   'Xcel Energy',
    'CVX':   'Chevron',
    'BAC':    'Bank of America',
    'BAH':     'Booze Allen Hamilton'
}
spark = get_spark("stocks")
#grab yfinance data 
ticker_lst =['PL=F', 'GC=F','SI=F','HG=F','PA=F', 'CL=F', 'BZ=F', 'XEL', 'CVX', 'BAC', 'BAH']
dt = yf.download(ticker_lst, start='2020-01-01', group_by='ticker')
#Download historical data for the last year
dt = pd.DataFrame(data=dt)
dt_f = dt.reset_index()
dt_f.columns = ['_'.join(col).strip() for col in dt_f.columns.values] #transform white space to underscore.
dt_f.columns = ["".join(col).replace('=','_') for col in dt_f.columns.values] #change '=' to underscore.
dt_f = np.round(dt_f, decimals=2)
        #print(dt_f.columns)

df_spark = spark.createDataFrame(dt_f)
df_spark = df_spark.withColumn('DateKey', F.date_format(F.col('Date_'),'yyyyMMdd').cast("int"))
df_spark = df_spark.withColumn('Date_',F.date_format(F.col('Date_'),'yyyy-MM-dd'))
df_spark.printSchema()
df_spark.show()
#load into delta table
df_spark.write.mode("overwrite").format("delta").option("inferSchema","true").saveAsTable("stocks_2")


C:\Users\test\AppData\Local\Temp/ipykernel_22720/1511851613.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dt = yf.download(ticker_lst, start='2020-01-01', group_by='ticker')
[*********************100%***********************]  11 of 11 completed


root
 |-- Date_: string (nullable = true)
 |-- BZ_F_Open: double (nullable = true)
 |-- BZ_F_High: double (nullable = true)
 |-- BZ_F_Low: double (nullable = true)
 |-- BZ_F_Close: double (nullable = true)
 |-- BZ_F_Volume: long (nullable = true)
 |-- PL_F_Open: double (nullable = true)
 |-- PL_F_High: double (nullable = true)
 |-- PL_F_Low: double (nullable = true)
 |-- PL_F_Close: double (nullable = true)
 |-- PL_F_Volume: double (nullable = true)
 |-- PA_F_Open: double (nullable = true)
 |-- PA_F_High: double (nullable = true)
 |-- PA_F_Low: double (nullable = true)
 |-- PA_F_Close: double (nullable = true)
 |-- PA_F_Volume: double (nullable = true)
 |-- GC_F_Open: double (nullable = true)
 |-- GC_F_High: double (nullable = true)
 |-- GC_F_Low: double (nullable = true)
 |-- GC_F_Close: double (nullable = true)
 |-- GC_F_Volume: double (nullable = true)
 |-- XEL_Open: double (nullable = true)
 |-- XEL_High: double (nullable = true)
 |-- XEL_Low: double (nullable = true)
 |-- XEL_Clos

In [3]:
from spark_session import get_spark
import pyspark.sql 
from pyspark.sql import functions as F
spark = get_spark('stop')
spark.stop()